# Imitation Learning
<img src='https://upload.wikimedia.org/wikipedia/commons/1/1f/Makak_neonatal_imitation.png?16484995326
01' width=1000/>

## Recommended Viewing

This tutorial only scratches the surface. For more, watch the imitation learning lecture of Berkeley's CS 182 (Sergey Levine): [Part 1](https://www.youtube.com/watch?v=kGc8jOy5_zY), [Part 2](https://www.youtube.com/watch?v=06uB13C5pxw), [Part 3](https://www.youtube.com/watch?v=a5wkzPa4fO4).

## Recap of Markov Decision Processes

Tut02 introduced stochastic Markov processes. What do we get if we add insentivised decision-making to a Markov chain?

A **Markov decision process** (MDP) is a tuple $\langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$:
- $\mathcal{S}$ is the state space, and $\mathcal{A}$ is the action space.
- $P(s' \mid s, a)$ is the transition function, with one transition matrix per action.
- $R(s, a)$ is the reward function.
- $\gamma \in [0, 1]$ is a discount factor that favors immediate rewards over future rewards.

A **policy** $\pi(a \mid s)$ chooses the actions. It can be deterministic ($a = \pi(s)$), or stochastic ($a \sim \pi(\cdot | s)$).

Under partial observability (a POMDP), the agent receives observations $o_t$ instead of states, and in general its policy must condition on the whole history $o_{1:t}$.
Like Gymnasium, we write $\pi(a_t \mid o_t)$ for both cases: if $o_t$ is the full state, this is an MDP; if not, we pretend $o_t$ is Markovian, which often works well in practice.

### The decision-making problem

![decision-making](images/decision-making.png)

Today, the policy is a neural network with parameters $\theta$, $\pi_\theta(a_t \mid o_t)$. It maps what the agent observes to what it does. **Where do good parameters come from?**

We implement this using Pytorch (see [helper notebook](PytorchDemo.ipynb)). We will discuss neural networks more deeply when when we learn about Deep Q-networks.

## What is Imitation Learning?

- In **reinforcement learning** (RL), the agent acts in the environment, receives observations and rewards, and learns from its own experience.
- In **imitation learning** (IL), the agent learns from **expert demonstrations**, which are records of what an expert, usually a human, did in each situation.

<p>
  <img src="images/rl-loop.png" alt="the agent-environment loop of RL" width="42%"/>
  &nbsp;&nbsp;
  <img src="images/monkey-imitation.jpg" alt="a newborn macaque imitates a human sticking out his tongue" width="52%"/>
</p>

_Question._ The imitating agent never sees a reward. What does it optimize? What does it need that an RL agent does not?

## Imitation Learning at Scale

Imitation is a main ingredient of today's foundation models:
- **Large language models** are pretrained by *next-token prediction*, predicting the next word that a human wrote. Next-token prediction is imitation learning.
- **Vision-language-action models** (VLAs) are the new state of the art in robot control. They predict the next robot action from camera images and a language instruction, after training on large collections of robot demonstrations.

<img src="images/openvla.jpg" alt="OpenVLA" width="700"/>

<p style="text-align: right; font-size: 0.8em">OpenVLA (<a href="https://openvla.github.io/">Kim et al., 2024</a>)</p>

### Modern(ish) imitation learning

Robot policies trained largely by imitating human demonstrations.

In [ ]:
from IPython.display import YouTubeVideo

# Toyota Research Institute teaches robots new behaviors from demonstrations
YouTubeVideo("w-CGSQAO5-Q", start=42, width=800, height=450)

In [ ]:
# Physical Intelligence's pi0.5, a VLA that works in homes it has never seen
YouTubeVideo("Zn8yMaepzVk", start=17, width=800, height=450)

## Behavior Cloning

The simplest form of imitation learning is **behavior cloning** (BC). It treats the demonstrations as a labeled dataset and fits the policy with supervised learning.

![behavior cloning](images/behavior-cloning.png)

Given demonstrations $\mathcal{D} = \{(o_i, a_i)\}_{i=1}^N$ of observations and the expert's actions:
- For **discrete actions** (classification), maximize the log-likelihood of the expert's actions, $\max_\theta \sum_i \log \pi_\theta(a_i \mid o_i)$. This is the cross-entropy loss.
- For **continuous actions** (regression), minimize the squared error, $\min_\theta \sum_i \lVert \pi_\theta(o_i) - a_i \rVert^2$. This is equivalent to maximizing the log-likelihood of a Gaussian policy with a fixed variance.

Everything we know from supervised learning applies: datasets, train/test splits, losses, overfitting.

_Question._ Is a good classifier also a good policy?

## Running Example, Taxi

The [Taxi environment](https://gymnasium.farama.org/environments/toy_text/taxi/) from Gymnasium:
- A $5 \times 5$ grid with walls and four marked locations: R, G, Y, and B.
- The taxi must drive to the passenger, pick them up, drive to the destination, and drop them off.
- 6 actions: south, north, east, west, pickup, and dropoff.
- Reward $-1$ per step, $+20$ for a delivery, and $-10$ for an illegal pickup or dropoff.
- $25 \cdot 5 \cdot 4 = 500$ states, one for each taxi position $\times$ passenger location (one of 4, or in the taxi) $\times$ destination.

In [ ]:
import gymnasium as gym
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm

from utils import (BC_COLOR, DAGGER_COLOR, AUG_COLOR, EXPERT_COLOR, MUTED, TAXI_ACTIONS, TaxiExpert,
                   WindingRoadEnv, make_taxi, plot_augmentation, plot_cars, plot_data_curve, plot_road,
                   plot_success_vs_labels, plot_trajectories, taxi_frames)

taxi_env = make_taxi(render_mode="rgb_array")

obs, info = taxi_env.reset(seed=0)
print("observation:", obs)
print("decoded (taxi row, taxi col, passenger, destination):", taxi_env.unwrapped.decode(obs))
media.show_image(taxi_env.render(), width=300)

### The expert

We need demonstrations. Hiring human taxi drivers is expensive, so our expert is a **planner**. Taxi is deterministic, and we know its transition function $s' = f(s, a)$, so we can compute an optimal driver exactly with **value iteration** ([Bellman, 1957](https://www.rand.org/pubs/papers/P1066.html), [Sutton & Barto, 2018, Sec. 4.4](http://incompleteideas.net/book/the-book-2nd.html)), a dynamic programming algorithm. 

We will discuss value iteration later in the course. For now, we assume that we can find a table telling us the optimal action at every state. For imitation learning, however, the expert is a black box that we can only **query**. We give it a state, and it returns an action.


In [ ]:
taxi_expert = TaxiExpert(taxi_env)

obs, info = taxi_env.reset(seed=0)
print("expert action at the first state:", TAXI_ACTIONS[taxi_expert(obs)])
media.show_video(taxi_frames(taxi_env, taxi_expert, seed=0), fps=3, width=300)

## Collecting Demonstrations

A **trajectory** is the sequence of states and actions that a policy experiences in one episode, $\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$, where $a_t = \pi(s_t)$. We run the expert, record its trajectories, and keep the (state, action) pairs.

In [ ]:
def run_episode(env, policy, seed=None, options=None):
    # returns the visited observations, the actions taken, the return, and the final info
    obs, info = env.reset(seed=seed, options=options)
    observations, actions, total_reward = [], [], 0.0
    while True:
        action = policy(obs)
        observations.append(obs)
        actions.append(action)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            return observations, actions, total_reward, info


def collect_demos(env, expert, n_episodes, seed=0):
    observations, actions = [], []
    for i in range(n_episodes):
        obs, acts, _, _ = run_episode(env, expert, seed=seed + i)
        observations += obs
        actions += acts
    return observations, actions

train_obs, train_actions = collect_demos(taxi_env, taxi_expert, n_episodes=50, seed=0)
test_obs, test_actions = collect_demos(taxi_env, taxi_expert, n_episodes=200, seed=10_000)

print(f"training set: {len(train_obs)} pairs from 50 episodes ({len(set(train_obs))} distinct states)")
print(f"test set:     {len(test_obs)} pairs from 200 episodes")

We keep duplicate pairs. States that the expert visits often should weigh more, so that the dataset follows the expert's state distribution.

### Representing the state

_Question._ How should we represent the state? Does it even matter?

A Taxi observation is a single integer in $\{0, \dots, 499\}$. As a number, it is meaningless, since states 17 and 18 are not "similar". We decode it into 7 features that a network can generalize over: the taxi's position, the passenger's position, whether the passenger is in the taxi, and the destination's position.

In [ ]:
LOCS = taxi_env.unwrapped.locs   # (row, col) of R, G, Y, B
IN_TAXI = 4                      # the passenger's location index while riding


def taxi_features(state):
    taxi_row, taxi_col, passenger, destination = taxi_env.unwrapped.decode(state)
    in_taxi = int(passenger == IN_TAXI)
    passenger_row, passenger_col = (taxi_row, taxi_col) if in_taxi else LOCS[passenger]
    destination_row, destination_col = LOCS[destination]
    return [taxi_row, taxi_col, passenger_row, passenger_col, in_taxi, destination_row, destination_col]


def featurize(states):
    return torch.tensor([taxi_features(s) for s in states], dtype=torch.float32)


X_train, y_train = featurize(train_obs), torch.tensor(train_actions)
X_test, y_test = featurize(test_obs), torch.tensor(test_actions)
print(f"state {train_obs[0]} -> features {taxi_features(train_obs[0])}")

## Training the Policy

The policy network is a multi-layer perceptron (MLP) with 7 features in and one score (*logit*) per action out. A softmax over the logits gives $\pi_\theta(a \mid s)$, and the cross-entropy loss of an example is exactly $-\log \pi_\theta(a \mid s)$ of the expert's action.

In [ ]:
def train_policy_net(X, y, out_dim, loss_fn, n_steps=2000, batch_size=64, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(),
                        nn.Linear(64, 64), nn.ReLU(),
                        nn.Linear(64, out_dim))
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for step in range(n_steps):
        idx = torch.randint(len(X), (batch_size,))   # a random minibatch
        loss = loss_fn(net(X[idx]), y[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return net


def accuracy(net, X, y):
    with torch.no_grad():
        return (net(X).argmax(dim=1) == y).float().mean().item()


taxi_net = train_policy_net(X_train, y_train, out_dim=6, loss_fn=nn.CrossEntropyLoss())
print(f"train accuracy: {accuracy(taxi_net, X_train, y_train):.3f}")
print(f"test accuracy:  {accuracy(taxi_net, X_test, y_test):.3f}")

The network agrees with the expert on almost 98% of the test states. For a classifier, that is excellent.

_Question._ Does that make it a good taxi driver?

## From Classifier to Policy

To act, the policy takes the highest-scoring action. Taxi has only 500 states, so we can run the network once on all of them and store its choices in a table. The learned policy *is* a table, just like the expert's.

In [ ]:
class TablePolicy:
    def __init__(self, net):
        with torch.no_grad():
            self.table = net(featurize(range(500))).argmax(dim=1).numpy()   # the network's choice at every state

    def __call__(self, state):
        return int(self.table[state])


bc_policy = TablePolicy(taxi_net)

### How good is the cloned driver?

There are only 300 possible start states (taxi position $\times$ passenger location $\times$ a different destination). Instead of sampling episodes, we test both drivers from **every** start state.

In [ ]:
START_STATES = np.flatnonzero(taxi_env.unwrapped.initial_state_distrib)


def evaluate_taxi(policy):
    # run the policy once from every start state, and return the per-start successes and returns
    episodes = [run_episode(taxi_env, policy, options={"start_state": s0}) for s0 in START_STATES]
    successes = np.array([info["success"] for _, _, _, info in episodes])
    returns = np.array([total_reward for _, _, total_reward, _ in episodes])
    return successes, returns


expert_success, expert_returns = evaluate_taxi(taxi_expert)
bc_success, bc_returns = evaluate_taxi(bc_policy)
print(f"expert:            success rate {expert_success.mean():.2f}   mean return {expert_returns.mean():6.1f}")
print(f"behavior cloning:  success rate {bc_success.mean():.2f}   mean return {bc_returns.mean():6.1f}")

Almost 98% accuracy, yet the cloned driver fails from about a quarter of the start states. A failure is expensive, because the episode runs until the 200-step time limit, at $-1$ per step.

### Watching it fail

In [ ]:
failed_starts = START_STATES[~bc_success]
print(f"BC fails from {len(failed_starts)} of {len(START_STATES)} start states. Slide through them, side by side with the expert:")


def show_failure(failure):
    s0 = failed_starts[failure]
    taxi_row, taxi_col, passenger, destination = taxi_env.unwrapped.decode(s0)
    print(f"start state {s0}: taxi at ({taxi_row}, {taxi_col}), passenger at {'RGYB'[passenger]}, "
          f"destination {'RGYB'[destination]}")
    media.show_videos({"expert": taxi_frames(taxi_env, taxi_expert, start_state=s0),
                       "behavior cloning": taxi_frames(taxi_env, bc_policy, start_state=s0, max_steps=20)},
                      fps=3, width=300)


widgets.interact(show_failure, failure=widgets.IntSlider(min=0, max=len(failed_starts) - 1,
                                                         continuous_update=False));


The cloned taxi gets stuck. It either drives into a wall again and again, or goes back and forth between two cells, until the time limit.

Two things go wrong:
1. **A mistake.** At some state, the network chooses a different action than the expert would.
2. **No recovery.** The policy is deterministic and memoryless. If an action leaves the state unchanged (driving into a wall), the next decision is identical, forever. The same holds for any loop of states.

_Question._ How could the agent escape such a loop?

## Compounding Errors

In supervised learning, mistakes are independent, and a classifier with error rate $\epsilon$ makes about $\epsilon N$ mistakes on $N$ test samples. A policy's mistakes are not independent, because **each action decides the next state**.

Suppose the policy errs with probability $\epsilon$ at every step of an episode of length $T$:
- The episode is free of mistakes with probability $(1 - \epsilon)^T$.
- In the worst case, as in Taxi, a single mistake ruins the rest of the episode, so a mistake at step $t$ costs up to $T - t$ steps. Summed over the episode, the expected cost grows as $O(\epsilon T^2)$, not as $O(\epsilon T)$ ([Ross & Bagnell, 2010](https://proceedings.mlr.press/v9/ross10a.html)).

In [ ]:
epsilon = 1 - accuracy(taxi_net, X_test, y_test)
T = taxi_expert.steps_to_go[START_STATES].mean()   # the expert's average episode length
print(f"per-step error epsilon = {epsilon:.3f},  episode length T = {T:.1f}")
print(f"(1 - epsilon)^T        = {(1 - epsilon) ** T:.2f}")
print(f"measured success rate  = {bc_success.mean():.2f}")

An error of about 2% per step becomes a quarter of the episodes failing. The measured success rate is a bit higher than $(1 - \epsilon)^T$, because not every mistake is fatal.

### Where are the mistakes?

In [ ]:
train_states = set(train_obs)
first_mistakes = []
for s0 in failed_starts:
    visited, _, _, _ = run_episode(taxi_env, bc_policy, options={"start_state": s0})
    first_mistakes.append(next(s for s in visited if bc_policy(s) != taxi_expert(s)))

unseen = np.mean([s not in train_states for s in first_mistakes])
print(f"in {unseen:.0%} of the failed episodes, the first mistake is at a state that is not in the training data")

The network fits the training states, and its mistakes happen where it must generalize. The features say where the taxi, passenger, and destination are, but not where the walls are. Next to a wall, the expert's action depends on whether the wall blocks the direct route, a detail that only data can teach.

### Just collect more data?

In [ ]:
n_demos_list = [10, 25, 50, 100, 200, 400]
metrics = {"test accuracy": [], "success rate": []}
for n_demos in tqdm(n_demos_list):
    obs, acts = collect_demos(taxi_env, taxi_expert, n_demos, seed=0)
    net = train_policy_net(featurize(obs), torch.tensor(acts), out_dim=6, loss_fn=nn.CrossEntropyLoss())
    metrics["test accuracy"].append(accuracy(net, X_test, y_test))
    metrics["success rate"].append(evaluate_taxi(TablePolicy(net))[0].mean())

plt.figure(figsize=(8, 4))
plot_data_curve(n_demos_list, metrics, title="Behavior cloning in Taxi")
plt.show()

In Taxi, more data is a complete fix. A few hundred demonstrations visit nearly every one of the 500 states, and the network can memorize the expert's table. Notice that the test accuracy is high long before the success rate is.

This does not scale. Real observations, such as camera images, never repeat exactly, so a learned policy must act in states that nobody demonstrated.

## Distributional Shift

![distributional shift](images/distribution-shift.png)

Supervised learning assumes that the training and test samples come from the same distribution. Behavior cloning breaks this assumption:
- We **train** on states from the expert's distribution, $p_{\text{data}}(o_t)$.
- We **test** on states from the learner's own distribution, $p_{\pi_\theta}(o_t)$.

Unless $\pi_\theta = \pi_{\text{expert}}$, we have $p_{\text{data}} \neq p_{\pi_\theta}$. Each mistake moves the learner to states the expert rarely visits, where it makes more mistakes, which take it even further away. The shift compounds.

Taxi hides this effect. The expert starts from every possible state, so $p_{\text{data}}$ covers almost all of them, and a mistake traps the taxi rather than taking it somewhere new. To see the shift, we need a world where the expert's demonstrations cover only a thin slice of the state space.

## Continuous Control on a Winding Road

We built a minimal driving environment for this purpose. A car drives at a constant speed along a winding road:
- The **observation** is the car's position and heading, $(x, y, \theta)$.
- The **action** is a continuous steering rate, $\omega \in [-2, 2]$.
- Random **wind gusts** disturb the steering at every step.
- The episode fails if the car leaves the road, and succeeds if it reaches the finish line at $x = 40$.

In [ ]:
road_env = WindingRoadEnv()
print("observation space:", road_env.observation_space)
print("action space:     ", road_env.action_space)

obs, info = road_env.reset(seed=0)
plt.figure(figsize=(12, 3.5))
plot_road(road_env)
plot_cars([obs], color=EXPERT_COLOR, length=1.5)
plt.text(obs[0] + 0.5, obs[1] - 0.3, "start", ha="center", va="top")
plt.show()


### The expert driver

Our expert is a simplified version of **pure pursuit** ([Coulter, 1992](https://www.ri.cmu.edu/publications/implementation-of-the-pure-pursuit-path-tracking-algorithm/)), a classic controller for following a path. At every step:
1. Pick a **target point on the center line of the road**, a fixed *lookahead* distance ahead of the car (1.5 units along $x$).
2. Compute the heading that would point the car straight at the target.
3. Steer in proportion to the difference between that heading and the car's current heading.

The target moves ahead with the car, so the car never reaches it. If the car drifts off-center, the target pulls it back to the center line. If the road bends, the target is already on the bend, so the car turns in time. Original pure pursuit steers along the circular arc that ends at the target point. For small heading errors, that is the same as steering in proportion to the error, as we do. The expert plays the role of a skilled human driver.


In [ ]:
def road_expert(obs, lookahead=1.5, gain=3.0):
    x, y, theta = obs
    target_y = road_env.center(x + lookahead)              # 1. the center of the road, slightly ahead
    target_heading = np.arctan2(target_y - y, lookahead)   # 2. the heading that points at it
    steer = gain * (target_heading - theta)                # 3. turn toward that heading
    return np.clip([steer], -road_env.max_steer, road_env.max_steer)


expert_runs = [run_episode(road_env, road_expert, seed=1000 + i) for i in range(20)]
print("expert success rate:", np.mean([info["success"] for *_, info in expert_runs]))

plt.figure(figsize=(12, 3.5))
plot_road(road_env)
plot_trajectories(expert_runs, color=EXPERT_COLOR, label="expert")
plt.show()

The expert never strays far from the center line. Its demonstrations cover only a thin tube inside the road.

### Behavior cloning with regression

The action is now a real number, so behavior cloning is a **regression** problem. We use the same network with a single output, trained with the squared error. The learner gets **2 demonstrations**.

In [ ]:
def road_features(observations):
    # scale (x, y, theta) to similar ranges, x from [0, length] to [-1, 1] and y by the road's amplitude
    x, y, theta = np.array(observations).T
    features = np.column_stack([2 * x / road_env.length - 1, y / road_env.amplitude, theta])
    return torch.tensor(features, dtype=torch.float32)


class RegressionPolicy:
    def __init__(self, net):
        self.net = net

    def __call__(self, obs):
        with torch.no_grad():
            return self.net(road_features([obs]))[0].numpy()


def train_road_bc(observations, actions, seed=0):
    y = torch.tensor(np.array(actions), dtype=torch.float32)
    net = train_policy_net(road_features(observations), y, out_dim=1, loss_fn=nn.MSELoss(),
                           n_steps=1500, batch_size=256, seed=seed)
    return RegressionPolicy(net)


demo_obs, demo_actions = collect_demos(road_env, road_expert, n_episodes=2, seed=0)
road_bc = train_road_bc(demo_obs, demo_actions)
print(f"{len(demo_obs)} expert labels")

### Evaluating BC

In [ ]:
def evaluate_road(policy, n_episodes=50):
    runs = [run_episode(road_env, policy, seed=10_000 + i) for i in range(n_episodes)]
    return np.mean([info["success"] for *_, info in runs]), runs


bc_road_success, bc_runs = evaluate_road(road_bc)
print(f"behavior cloning success rate: {bc_road_success:.2f}")

demo_runs = [run_episode(road_env, road_expert, seed=i) for i in range(2)]   # the 2 demonstrations
plt.figure(figsize=(12, 3.5))
plot_road(road_env)
plot_trajectories(bc_runs[:20], color=BC_COLOR, label="behavior cloning")
plot_trajectories(demo_runs, color=EXPERT_COLOR, label="demonstrations", linewidth=2, alpha=1)
plt.show()

This is the picture from the distributional shift slide, for real. The cloned driver follows the demonstrations for a while, drifts out of the tube, and once outside, it no longer knows how to steer back.

### Measuring the shift

In [ ]:
def imitation_error(policy, observations):
    # mean squared difference between the policy's actions and the expert's actions
    return np.mean([(policy(o) - road_expert(o)) ** 2 for o in observations])


expert_states = [o for run in expert_runs for o in run[0]]    # states from p_data
learner_states = [o for run in bc_runs for o in run[0]]       # states from p_pi
print(f"error on the expert's states:  {imitation_error(road_bc, expert_states):.3f}")
print(f"error on the learner's states: {imitation_error(road_bc, learner_states):.3f}")

On the states it actually visits, the learner's error is almost an order of magnitude larger than on the expert's states. The test error of supervised learning measures the wrong distribution.

## DAgger: Dataset Aggregation

Can we make $p_{\text{data}} = p_{\pi_\theta}$? Instead of being clever about the learner, be clever about the data, and **collect training data from the learner's distribution**. Running the learner gives us its states, but we still need labels, the expert's actions in those states.

**DAgger** ([Ross, Gordon & Bagnell, 2011](https://proceedings.mlr.press/v15/ross11a.html)):
1. Train $\pi_\theta$ on the expert demonstrations $\mathcal{D} = \{(o_1, a_1), \dots, (o_N, a_N)\}$.
2. Run $\pi_\theta$ to collect a dataset of observations $\mathcal{D}_\pi = \{o_1, \dots, o_M\}$.
3. Ask the expert to label $\mathcal{D}_\pi$ with actions.
4. Aggregate, $\mathcal{D} \leftarrow \mathcal{D} \cup \mathcal{D}_\pi$, and repeat from step 1.

The learner's mistakes are now in the training data, labeled with the expert's corrections. Under some assumptions, DAgger's cost grows only *linearly* with the horizon, $O(\epsilon T)$, where $\epsilon$ is the error on the learner's own distribution.

### DAgger on the winding road

We start from the same 2 demonstrations. Each round runs the learner for one episode and asks the expert to label every state it visited.

In [ ]:
def dagger(n_demos, n_iterations, seed=0):
    observations, actions = collect_demos(road_env, road_expert, n_demos, seed=100 * seed)
    policy = train_road_bc(observations, actions, seed=seed)                    # 1. train on the demos
    history = [(len(observations), evaluate_road(policy)[0])]
    for i in tqdm(range(n_iterations), desc="training", leave=False):
        new_obs, _, _, _ = run_episode(road_env, policy, seed=100 * seed + 50 + i)  # 2. run the learner
        new_actions = [road_expert(o) for o in new_obs]                             # 3. ask the expert
        observations, actions = observations + new_obs, actions + new_actions       # 4. aggregate
        policy = train_road_bc(observations, actions, seed=seed)                    # 1. retrain
        history.append((len(observations), evaluate_road(policy)[0]))
    return policy, history


dagger_policy, dagger_history = dagger(n_demos=2, n_iterations=6)
for n_labels, success_rate in dagger_history:
    print(f"{n_labels:5d} expert labels   success rate {success_rate:.2f}")

In [ ]:
_, dagger_runs = evaluate_road(dagger_policy)
plt.figure(figsize=(12, 3.5))
plot_road(road_env)
plot_trajectories(dagger_runs[:20], color=DAGGER_COLOR, label="DAgger")
plot_trajectories(demo_runs, color=EXPERT_COLOR, label="demonstrations", linewidth=2, alpha=1)
plt.show()

### DAgger or more demonstrations?

DAgger used more expert labels than plain behavior cloning. A fair comparison gives behavior cloning the same budget of labels, as extra demonstrations. We repeat both with 3 seeds (seed 0 of DAgger is the run above).

In [ ]:
results = {"behavior cloning": [], "DAgger": [dagger_history]}
for seed in tqdm(range(3), desc="random seed trials"):
    bc_curve = []
    for n_demos in tqdm([2, 4, 8, 16], desc="training BC", leave=False):
        obs, acts = collect_demos(road_env, road_expert, n_demos, seed=100 * seed)
        bc_curve.append((len(obs), evaluate_road(train_road_bc(obs, acts, seed=seed))[0]))
    results["behavior cloning"].append(bc_curve)
    if seed > 0:
        results["DAgger"].append(dagger(n_demos=2, n_iterations=6, seed=seed)[1])

plt.figure(figsize=(8, 4))
plot_success_vs_labels(results, title="The winding road")
plt.show()

With the same budget of expert labels, DAgger wins clearly. Every DAgger run drives without failures after two rounds of corrections, fewer than 900 labels. Behavior cloning with 4 times as many labels (16 demonstrations) still leaves the road occasionally, because the extra demonstrations mostly repeat what the expert has already shown, driving in the middle of the road.

_Question._ What's the catch?

- The expert must be available **during training**, to label new states on demand.
- Labeling is unnatural for humans. It is hard to say what you would have done in a situation that you did not drive yourself, one frame at a time.
- The learner drives badly while it collects data. In the real world, this can be dangerous.

## Data Augmentation

Another way to be clever about the data, without querying the expert online, is to **synthesize** the off-center states that the learner will visit and label them with corrective actions.

![data augmentation](images/data-augmentation.png)

NVIDIA's self-driving car ([Bojarski et al., 2016](https://arxiv.org/abs/1604.07316)) recorded human drivers with three cameras. The left and right cameras see the road as if the car had drifted sideways, so their images are labeled with the human's steering plus a correction back toward the center.

### Does it work?

In [ ]:
# NVIDIA's car, trained end-to-end from the augmented demonstrations
YouTubeVideo("qhUvQiKec2U", width=800, height=450)

### Augmenting the road demonstrations

We imitate NVIDIA's side cameras. For every demonstrated state, we add two synthetic ones.
- The **left camera** sees the car shifted 0.3 to its left and turned 0.15 rad counter-clockwise.
- The **right camera** sees the mirror image, shifted to the right and turned clockwise.

Each synthetic state needs a label. **We do not ask the expert** (that would be DAgger in disguise). Like Bojarski et al., 2016, we label it with the demonstrated steering plus a fixed correction, which would bring the car back onto the demonstrated path, pointing along it, within $\tau = 2$ seconds,
$$a_{\text{synthetic}} = a_{\text{demo}} - \frac{6 e}{v \tau^2} - \frac{4 \phi}{\tau},$$
where $e$ is the sideways shift (positive to the left), $\phi$ is the turn (positive counter-clockwise), and $v$ is the car's speed. The return time $\tau$ is our choice, and NVIDIA also used 2 seconds. A smaller $\tau$ corrects more sharply. The only expert labels are still those of the 2 demonstrations.

<p style="font-size: 0.8em">Where does the formula come from? Near the demonstrated path, the sideways offset accelerates in proportion to the extra steering $u$, $\ddot e = v u$. The formula is the first step of the gentlest steering (the smallest $\int u^2 \, dt$) that brings both $e$ and $\phi$ to zero after $\tau$ seconds.</p>


In [ ]:
def side_camera(states, steering, shift, turn, tau=2.0):
    # the view of a camera shifted `shift` to the car's left and turned `turn` counter-clockwise
    # (negative values shift it to the right and turn it clockwise), labeled with the corrected steering
    x, y, theta = states.T
    views = np.column_stack([x - shift * np.sin(theta), y + shift * np.cos(theta), theta + turn])
    correction = -6 * shift / (road_env.speed * tau ** 2) - 4 * turn / tau   # back on the path in tau seconds
    labels = np.clip(steering + correction, -road_env.max_steer, road_env.max_steer)
    return views, labels, correction


demos, demo_steering = np.array(demo_obs), np.array(demo_actions)
left_views, left_labels, left_correction = side_camera(demos, demo_steering, shift=0.3, turn=0.15)
right_views, right_labels, right_correction = side_camera(demos, demo_steering, shift=-0.3, turn=-0.15)
augmented_obs = np.concatenate([demos, left_views, right_views])
augmented_actions = np.concatenate([demo_steering, left_labels, right_labels])
print(f"{len(demos)} demonstrated states + {len(augmented_obs) - len(demos)} synthetic states")
print(f"corrections: left camera {left_correction:+.3f}, right camera {right_correction:+.3f}")


### A closer look at the synthetic states

Each demonstrated state (black) gets two synthetic copies (green), as seen by the side cameras. A label is a single steering rate. To show it, each state's dots trace where the car would go if it held its steering for 1 second, one dot per time step of 0.1 seconds. The black dots follow the demonstrated steering. The green dots follow the labels, which add the correction. The left camera's copies bend to the right of their black neighbors, and the right camera's copies bend to the left, back toward the demonstration.


In [ ]:
first_demo = demos[:len(demo_runs[0][0])]
shown = np.flatnonzero((first_demo[:, 0] > 3) & (first_demo[:, 0] < 9))[::12]   # a few states of the first demo

plt.figure(figsize=(12, 5))
plot_road(road_env)
plot_trajectories(demo_runs[:1], color=EXPERT_COLOR, label="demonstration", linewidth=1, alpha=0.4)
plot_augmentation(road_env, demos[shown], demo_steering[shown], [left_views[shown], right_views[shown]],
                  [left_labels[shown], right_labels[shown]], duration=1.0)
plt.show()


### Training on the augmented data


In [ ]:
augmented_policy = train_road_bc(augmented_obs, augmented_actions)
augmented_success, augmented_runs = evaluate_road(augmented_policy)
print(f"success rate from the same 2 demonstrations: BC {bc_road_success:.2f}, BC + augmentation {augmented_success:.2f}")

plt.figure(figsize=(12, 3.5))
plot_road(road_env)
plot_trajectories(augmented_runs[:20], color=AUG_COLOR, label="data augmentation")
plot_trajectories(demo_runs, color=EXPERT_COLOR, label="demonstrations", linewidth=2, alpha=1)
plt.show()


### Following forest trails

The same trick taught a quadcopter to follow forest trails ([Giusti et al., 2016](https://doi.org/10.1109/LRA.2015.2509024)). A hiker wore three cameras. Images from the left-facing camera were labeled "turn right", from the right-facing camera "turn left", and from the forward camera "go straight".

In [ ]:
YouTubeVideo("umRdt3zGgpU", start=104, width=800, height=450)

## Multi-Task Learning

<img src="images/multi-task.png" alt="a policy for one goal vs. a goal-conditioned policy" width="650"/>

Learning many tasks at once can make imitation *easier*. A **goal-conditioned** policy $\pi_\theta(a \mid s, p)$ receives the goal $p$ as an additional input:
- Demonstrations for different goals visit different states, so the data covers more of the state space.
- A state that is off-course for one goal is on-course for another.

Our taxi policy is already goal-conditioned, since the passenger's location and the destination are part of its input. Moreover, *any* trajectory is an expert demonstration of how to reach the state where it ended. Relabeling each trajectory with its final state as the goal turns even imperfect data into useful demonstrations ([Ghosh et al., 2019](https://arxiv.org/abs/1912.06088)).

## Multimodal Behavior

<img src="images/multimodal-tree.png" alt="two ways around a tree" width="200" align="right"/>

There can be **more than one correct action** in a state. To pass a tree, the expert sometimes swerves left and sometimes right. Both are fine. Their average is not.

In [ ]:
rng = np.random.default_rng(0)
steering = np.concatenate([rng.normal(-1.0, 0.15, 500),    # demonstrations that pass on the left
                           rng.normal(+1.0, 0.15, 500)])   # demonstrations that pass on the right
mse_prediction = steering.mean()                           # the prediction that minimizes the squared error

plt.figure(figsize=(8, 3))
plt.hist(steering, bins=60, color=MUTED, label="expert steering in front of the tree")
plt.axvline(mse_prediction, color=BC_COLOR, linewidth=2, label=f"squared-error regression ({mse_prediction:.2f})")
plt.xlabel("steering")
plt.ylabel("count")
plt.legend(frameon=False)
plt.show()

Squared-error regression predicts the *mean* action, straight into the tree. The policy needs a distribution with several modes:
- **Discretize** the actions and use a categorical policy, as in Taxi. A softmax can put half of its mass on "left" and half on "right".
- With a **mixture of Gaussians**, the network outputs the means, variances, and weights of several Gaussians.
- **Latent variable models**, such as conditional variational autoencoders, where a latent variable selects the mode.
- **Diffusion models**, which generate an action by iteratively denoising random noise.

## History-Dependent Policies

So far, the policy sees only the current observation. When the process is not Markovian, as is often the case in POMDPs, the expert's action may depend on the **history** of observations.
- **Sequence models**, such as LSTMs and Transformers, can condition the policy on the history, $\pi_\theta(a_t \mid o_{1:t})$.
- A taxi with memory could notice that its last 10 moves into the wall changed nothing.

More information is not always better. With history, a policy can latch on to **spurious correlations**, a problem known as *causal confusion* ([de Haan, Jayaraman & Levine, 2019](https://arxiv.org/abs/1905.11979)). A driving policy that sees the dashboard's brake indicator learns that "indicator on" predicts braking. It then copies its own past action, instead of reacting to the obstacle that made the expert brake.

## Limitations of Imitation Learning

<img src="images/imitation-learning-humanoid-robot.png" alt="the Atlas humanoid robot" height="300" align="right"/>

- **Data must be given.** This is an unavoidable bottleneck. Human demonstrations are slow to collect, and deep learning works best when data is plentiful.
- **Humans are not good at providing actions to agents.** Demonstrating requires specialized interfaces, such as teleoperation rigs.

<p>
  <img src="images/imitation-learning-driving-example.png" alt="the forest trail quadcopter's camera view and steering output" height="140"/>
  &nbsp;&nbsp;
  <img src="images/imitation-learning-remote-controller.png" alt="a quadcopter remote controller" height="140"/>
  &nbsp;&nbsp;
  <img src="images/imitation-learning-quadcopter-diagram.png" alt="the four rotors of a quadcopter" height="140"/>
</p>

<p style="font-size: 0.8em">The forest trail data came from a hiker's cameras, not from a pilot's steering. A pilot flies a quadcopter with two sticks, not by setting the thrust of each of its four rotors. Nobody can directly demonstrate the joint commands of a humanoid robot with dozens of joints.</p>

- **The policy is limited by the expert.** Are there better behaviors that the expert never thought of?
- **The policy is static.** Can we build policies that keep learning and improving?

Reinforcement learning answers the last two questions. The agent learns from its own experience and rewards, and can surpass any demonstrator. The price is what imitation spared us today, namely lots of interaction and learning from rewards instead of from labeled actions.

## Conclusion

- **Behavior cloning** turns decision making into supervised learning, with classification for discrete actions and regression for continuous ones.
- **An accurate classifier is not necessarily a good policy.** Errors compound over the horizon, up to $O(\epsilon T^2)$. In Taxi, almost 98% accuracy still failed about a quarter of the episodes.
- Under **distributional shift**, the learner trains on the expert's states but is tested on its own. On the winding road, its error on its own states was almost an order of magnitude larger.
- The fixes change the **data**: more demonstrations (only in small state spaces), **DAgger** (expert labels for the learner's states), and **data augmentation** (synthetic corrections).
- Beyond behavior cloning: goal conditioning, multimodal action distributions, and history-dependent policies.

### Further reading

- [A Reduction of Imitation Learning and Structured Prediction to No-Regret Online Learning](https://arxiv.org/abs/1011.0686) (Ross, Gordon & Bagnell, 2011), the DAgger paper.
- [Generative Adversarial Imitation Learning](https://arxiv.org/abs/1606.03476) (Ho & Ermon, 2016), imitation without supervised learning of actions.
- [OpenVLA: An Open-Source Vision-Language-Action Model](https://arxiv.org/abs/2406.09246) (Kim et al., 2024).